# Complete Journey weekly panel for ICDN

Step-by-step construction of the observation unit

`(store, product, week) → (price, units, promo, metadata)`

We do **not** call `builder.run()`. Each cell invokes one method so we can inspect the intermediate objects.

**Files used:** `transaction_data.csv`, `product.csv`, `causal_data.csv`.  
**Not used:** `coupon.csv`, `coupon_redempt.csv`, `campaign_desc.csv`, `campaign_table.csv`, `hh_demographic.csv`.

Unlike M5, there is **no shelf-price panel**. We keep only store-product-weeks with an observed purchase. Do not impute `units = 0` or forward-fill price.

`units` are purchases by panel households, not store-wide sales.

`WEEK_NO` is already sequential (`1…102`). Do not remap it.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from dunn import DunnConfig, DunnSampleSelection, DunnWeeklyPanelBuilder

DATA_DIR = PROJECT_ROOT / "data" / "dunnhumby"
OUT_DIR = DATA_DIR / "panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("Exists      :", DATA_DIR.exists())
print("Files       :", sorted(p.name for p in DATA_DIR.glob("*.csv")) if DATA_DIR.exists() else "—")

PROJECT_ROOT: /home/thebigmonster/Github/nn-elasticity-additional-work
DATA_DIR    : /home/thebigmonster/Github/nn-elasticity-additional-work/data/dunnhumby
Exists      : True
Files       : ['campaign_desc.csv', 'campaign_table.csv', 'causal_data.csv', 'coupon.csv', 'coupon_redempt.csv', 'hh_demographic.csv', 'product.csv', 'transaction_data.csv']


## 0. Config and builder

Store, commodity and SKU rules are stored now but applied only on `week_id <= 51`.

In [2]:
config = DunnConfig(
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    selection_cutoff=51,
    min_store_weeks=45,
    n_core_stores=20,
    min_stores=5,
    min_weeks=35,
    min_unique_prices=8,
    min_price_cv=0.03,
    min_products_in_group=10,
    n_candidate_skus=30,
    n_skus=10,
    min_promo_coverage=0.2,
)

builder = DunnWeeklyPanelBuilder(config)

## 1. Product master

`COMMODITY_DESC` → ICDN `category`.  
`BRAND` is mainly Private / National, not a commercial brand name.

In [3]:
products = builder.load_products()

(92353, 7)
   PRODUCT_ID MANUFACTURER    DEPARTMENT     BRAND            COMMODITY_DESC           SUB_COMMODITY_DESC CURR_SIZE_OF_PRODUCT
0       25671            2       GROCERY  National                  FRZN ICE          ICE - CRUSHED/CUBED                22 LB
1       26081            2  MISC. TRANS.  National  NO COMMODITY DESCRIPTION  NO SUBCOMMODITY DESCRIPTION                     
2       26093           69        PASTRY   Private                     BREAD         BREAD:ITALIAN/FRENCH                     
3       26190           69       GROCERY   Private      FRUIT - SHELF STABLE                  APPLE SAUCE                50 OZ
4       26355           69       GROCERY   Private             COOKIES/CONES            SPECIALTY COOKIES                14 OZ


## 2. Transactions and audit

Keep these counts for the appendix. Household keys stay out of ICDN.

In [4]:
tx = builder.load_transactions()
print(tx.shape)
tx.head()

rows                2595732
households             2500
stores                  582
products              92339
weeks                   102
week_min                  1
week_max                102
quantity_le_0         14466
sales_value_le_0      18850
missing_product           0
missing_store             0
missing_week              0
dtype: int64
(2595732, 12)


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1.0000,1.3900,364,-0.6000,1631,1,0.0000,0.0000
1,2375,26984851472,1,1033142,1.0000,0.8200,364,0.0000,1631,1,0.0000,0.0000
2,2375,26984851472,1,1036325,1.0000,0.9900,364,-0.3000,1631,1,0.0000,0.0000
3,2375,26984851472,1,1082185,1.0000,1.2100,364,0.0000,1631,1,0.0000,0.0000
4,2375,26984851472,1,8160430,1.0000,1.5000,364,-0.3900,1631,1,0.0000,0.0000


## 3. Drop impossible rows

For log-demand / log-price we need `q > 0` and `SALES_VALUE > 0`. No winsorizing: huge `QUANTITY` is typical of weighted/bulk UPCs and is handled later by the commodity screen.

In [5]:
tx_valid = builder.drop_invalid_transactions(tx)
del tx
tx_valid.head()

Kept 2,576,815 / 2,595,732 rows (99.27%)


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1.0000,1.3900,364,-0.6000,1631,1,0.0000,0.0000
1,2375,26984851472,1,1033142,1.0000,0.8200,364,0.0000,1631,1,0.0000,0.0000
2,2375,26984851472,1,1036325,1.0000,0.9900,364,-0.3000,1631,1,0.0000,0.0000
3,2375,26984851472,1,1082185,1.0000,1.2100,364,0.0000,1631,1,0.0000,0.0000
4,2375,26984851472,1,8160430,1.0000,1.5000,364,-0.3900,1631,1,0.0000,0.0000


## 4. Monetary variables

Primary value is retailer receipts `SALES_VALUE`.

- `customer_spend = SALES_VALUE − |COUPON_DISC|`
- `regular_value = SALES_VALUE + |RETAIL_DISC| + |COUPON_MATCH_DISC|`

`has_retail_discount` and `has_coupon` are diagnostics, **not** `on_promo`.

In [6]:
tx_valid = builder.construct_value_variables(tx_valid)
tx_valid[
    [
        "QUANTITY",
        "SALES_VALUE",
        "customer_spend",
        "regular_value",
        "has_retail_discount",
        "has_coupon",
    ]
].head()

,QUANTITY,SALES_VALUE,customer_spend,regular_value,has_retail_discount,has_coupon
0,1.0000,1.3900,1.3900,1.9900,1,0
1,1.0000,0.8200,0.8200,0.8200,0,0
2,1.0000,0.9900,0.9900,1.2900,1,0
3,1.0000,1.2100,1.2100,1.2100,0,0
4,1.0000,1.5000,1.5000,1.8900,1,0


## 5. Line-quantity stats (bulk / weighted UPCs)

In [7]:
builder.line_qty_stats = builder.line_quantity_stats(tx_valid)
builder.line_qty_stats.sort_values("p99_line_qty", ascending=False).head(15)

,PRODUCT_ID,median_line_qty,p99_line_qty,max_line_qty,product_code
57116,6544236,"20,051.0000","39,507.8500","85,055.0000",6544236
54662,5716076,"23,855.0000","29,955.5000","30,080.0000",5716076
54496,5668996,"6,421.5000","27,032.7700","29,945.0000",5668996
57002,6534178,"10,568.0000","25,011.0000","89,638.0000",6534178
3562,397896,"11,144.0000","24,210.0300","25,567.0000",397896
56963,6533889,"12,953.0000","23,805.2000","32,035.0000",6533889
44856,1426702,"13,703.0000","23,387.2600","25,652.0000",1426702
44633,1404121,"9,049.5000","23,142.7500","25,750.0000",1404121
56997,6534166,"11,680.5000","22,485.1000","36,140.0000",6534166
56066,6410462,"10,096.5000","22,107.6700","22,451.0000",6410462


## 6. Aggregate to store × product × week

`household × basket × product` → `store × product × week`.

In [8]:
weekly = builder.aggregate_to_weekly(tx_valid)
del tx_valid
print(weekly.shape)
weekly.head()

(2355132, 14)


,STORE_ID,PRODUCT_ID,WEEK_NO,units,sales_value,customer_spend,regular_value,retail_discount,coupon_discount,coupon_match,n_baskets,n_households,retail_discount_events,coupon_events
0,1,480014,5,"7,249.0000",15.0000,15.0000,15.7200,0.7200,0.0000,0.0000,1,1,1,0
1,1,480014,6,"2,438.0000",5.0000,5.0000,5.2400,0.2400,0.0000,0.0000,1,1,1,0
2,1,718226,5,1.0000,7.4000,7.4000,7.4000,0.0000,0.0000,0.0000,1,1,0,0
3,1,6903760,14,1.0000,2.4900,2.4900,2.4900,0.0000,0.0000,0.0000,1,1,0,0
4,2,480415,100,1.0000,1.7900,1.7900,1.7900,0.0000,0.0000,0.0000,1,1,0,0


## 7. Quantity-weighted prices

$$P = \sum \mathrm{SALES\_VALUE} \big/ \sum \mathrm{QUANTITY}$$

Not the mean of line unit prices. `price_customer` and `price_regular` stay on the master for robustness.

In [9]:
weekly = builder.compute_weekly_prices(weekly)
weekly[["price", "price_customer", "price_regular", "units"]].head()

               price  price_customer  price_regular          units
count 2,355,132.0000  2,355,132.0000 2,355,132.0000 2,355,132.0000
mean          2.5017          2.4910         2.9050       110.6864
std           2.7618          2.7551         3.0559     3,035.3679
min           0.0017        -12.9900         0.0018         1.0000
1%            0.2000          0.2000         0.2500         1.0000
5%            0.4900          0.4700         0.5400         1.0000
50%           1.9900          1.9900         2.2900         1.0000
95%           6.4600          6.4000         7.4400         3.0000
99%          12.2900         12.2000        13.8900         6.0000
max         499.9900        499.9900       549.9900   262,301.0000


,price,price_customer,price_regular,units
0,0.0021,0.0021,0.0022,"7,249.0000"
1,0.0021,0.0021,0.0021,"2,438.0000"
2,7.4000,7.4000,7.4000,1.0000
3,2.4900,2.4900,2.4900,1.0000
4,1.7900,1.7900,1.7900,1.0000


## 8. Product metadata

Products without a match must remain visible, not silently dropped.

In [10]:
weekly = builder.attach_product_metadata(weekly)
weekly[["PRODUCT_ID", "COMMODITY_DESC", "BRAND", "DEPARTMENT"]].head()

Missing product metadata: 0.0


,PRODUCT_ID,COMMODITY_DESC,BRAND,DEPARTMENT
0,480014,COUPON/MISC ITEMS,Private,KIOSK-GAS
1,480014,COUPON/MISC ITEMS,Private,KIOSK-GAS
2,718226,COUPON/MISC ITEMS,National,MISC SALES TRAN
3,6903760,LAXATIVES,Private,DRUG GM
4,480415,VEGETABLES SALAD,National,PRODUCE


## 9. ICDN identifiers

No `week_id` remap: `WEEK_NO` is already `1…102`.

In [11]:
weekly = builder.to_icdn_identifiers(weekly)

weeks = np.sort(weekly["week_id"].unique())
print("n weeks:", len(weeks), "range:", weeks.min(), "→", weeks.max())
print("max gap in week_id:", np.diff(weeks).max() if len(weeks) > 1 else None)
weekly.head()

week_id range: 1 → 102
n weeks: 102 range: 1 → 102
max gap in week_id: 1


,store_code,product_code,week_id,units,sales_value,customer_spend,regular_value,retail_discount,coupon_discount,coupon_match,n_baskets,n_households,retail_discount_events,coupon_events,price,price_customer,price_regular,retail_discount_share,coupon_share,brand,department,style,category,sub_commodity,size
0,1,480014,5,"7,249.0000",15.0000,15.0000,15.7200,0.7200,0.0000,0.0000,1,1,1,0,0.0021,0.0021,0.0022,0.0458,0.0000,69,KIOSK-GAS,Private,COUPON/MISC ITEMS,GASOLINE-REG UNLEADED,
1,1,480014,6,"2,438.0000",5.0000,5.0000,5.2400,0.2400,0.0000,0.0000,1,1,1,0,0.0021,0.0021,0.0021,0.0458,0.0000,69,KIOSK-GAS,Private,COUPON/MISC ITEMS,GASOLINE-REG UNLEADED,
2,1,718226,5,1.0000,7.4000,7.4000,7.4000,0.0000,0.0000,0.0000,1,1,0,0,7.4000,7.4000,7.4000,0.0000,0.0000,4,MISC SALES TRAN,National,COUPON/MISC ITEMS,MISC SALES TRANS,
3,1,6903760,14,1.0000,2.4900,2.4900,2.4900,0.0000,0.0000,0.0000,1,1,0,0,2.4900,2.4900,2.4900,0.0000,0.0000,69,DRUG GM,Private,LAXATIVES,LAXATIVES,3OZ 211565
4,2,480415,100,1.0000,1.7900,1.7900,1.7900,0.0000,0.0000,0.0000,1,1,0,0,1.7900,1.7900,1.7900,0.0000,0.0000,529,PRODUCE,National,VEGETABLES SALAD,VARIETY LETTUCE,12 CT


## 10. Save the transaction master

Observed purchases only. **Do not delete this file.** Causal flags are not on it yet.

In [12]:
builder.weekly = builder.save_transaction_master(weekly)
builder.weekly.head()

Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/dunnhumby/panel/dunnhumby_weekly_transaction_master.parquet


,store_code,product_code,week_id,units,sales_value,customer_spend,regular_value,retail_discount,coupon_discount,coupon_match,n_baskets,n_households,retail_discount_events,coupon_events,price,price_customer,price_regular,retail_discount_share,coupon_share,brand,department,style,category,sub_commodity,size
0,2602,3843566,1,2.0000,5.0000,5.0000,5.9800,0.9800,0.0000,0.0000,1,1,1,0,2.5000,2.5000,2.9900,0.1639,0.0000,2,VIDEO RENTAL,National,UNKNOWN,VIDEO RENTALS,
1,27,447776,1,1.0000,0.9900,0.9900,0.9900,0.0000,0.0000,0.0000,1,1,0,0,0.9900,0.9900,0.9900,0.0000,0.0000,69,GROCERY,Private,COFFEE,NON DAIRY CREAMER: DRY,11 OZ
2,27,482616,1,2.0000,1.5800,1.5800,1.5800,0.0000,0.0000,0.0000,1,1,0,0,0.7900,0.7900,0.7900,0.0000,0.0000,905,DRUG GM,National,CANDY - PACKAGED,SEASONAL CANDY BOX-CHOCOLATE,1.75 OZ
3,27,518145,1,1.0000,7.9900,7.9900,7.9900,0.0000,0.0000,0.0000,1,1,0,0,7.9900,7.9900,7.9900,0.0000,0.0000,5065,DRUG GM,National,FIRST AID PRODUCTS,MISC. FIRST AID PRODUCTS,.31 OZ
4,27,552267,1,1.0000,1.4900,1.4900,1.4900,0.0000,0.0000,0.0000,1,1,0,0,1.4900,1.4900,1.4900,0.0000,0.0000,870,DRUG GM,National,EASTER,EASTER EGG COLORING,


## 11. Selection window (no lookahead)

Weeks 52–102 do not enter store / SKU / category choice.

In [13]:
selection = builder.selection_sample()
print(selection["week_id"].min(), "→", selection["week_id"].max())
print("rows:", len(selection))

weeks: 1 102 cutoff: 51
1 → 51
rows: 1052850


## 12. Core stores

~2,500 households cannot fill hundreds of stores. Keep stores with ≥45 weeks, then the 20 most active. This is an initial universe, not necessarily the final store set.

In [14]:
core_stores = builder.select_core_stores(selection)
selection_core = selection[selection["store_code"].isin(core_stores)].copy()
print("core rows:", len(selection_core))

core stores: ['367', '406', '361', '356', '381', '32004', '292', '427', '31782', '375', '372', '319', '321', '401', '333', '369', '422', '388', '439', '327']
core rows: 322460


## 13. Product diagnostics

`coverage_rate` = share of **core store-weeks** with an observed purchase of that UPC.

Do **not** call this `positive_rate`: we do not observe retailer zeros.

In [15]:
product_stats = builder.compute_product_stats(selection_core)
product_stats.head(20)

Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/dunnhumby/panel/dunnhumby_product_diagnostics.csv


,product_code,category,sub_commodity,brand,style,department,n_obs,n_stores,n_weeks,total_units,unique_prices,mean_price,std_price,mean_baskets,mean_households,discount_rate,coupon_rate,price_cv,coverage_rate,median_line_qty,p99_line_qty,max_line_qty
7314,1082185,TROPICAL FRUIT,BANANAS,2,National,PRODUCE,923,20,51,"4,443.0000",664,0.9643,0.2611,4.6804,4.2904,0.2449,0.0000,0.2708,0.9049,1.0000,2.0000,9.0000
2805,1029743,FLUID MILK PRODUCTS,FLUID MILK WHITE ONLY,69,Private,GROCERY,746,20,51,"2,349.0000",83,2.3853,0.2752,2.7064,2.4276,0.2413,0.0040,0.1154,0.7314,1.0000,3.0000,10.0000
36058,995242,FLUID MILK PRODUCTS,FLUID MILK WHITE ONLY,69,Private,GROCERY,648,20,51,"2,586.0000",64,1.5017,0.4362,2.4583,2.2485,0.5725,0.0000,0.2904,0.6353,1.0000,8.0000,14.0000
34402,981760,EGGS,EGGS - X-LARGE,69,Private,GROCERY,628,20,51,"1,492.0000",61,1.1075,0.1652,1.9745,1.9108,0.5732,0.0016,0.1491,0.6157,1.0000,3.0000,10.0000
9139,1106523,FLUID MILK PRODUCTS,FLUID MILK WHITE ONLY,69,Private,GROCERY,618,20,51,"1,432.0000",45,2.3891,0.2824,2.0615,1.8673,0.2120,0.0049,0.1182,0.6059,1.0000,3.0000,12.0000
36098,995785,PEPPERS-ALL,PEPPERS GREEN BELL,2,National,PRODUCE,531,20,51,"1,293.0000",39,0.7341,0.1833,1.5556,1.5348,0.0847,0.0000,0.2497,0.5206,1.0000,5.0000,12.0000
30969,951590,BAKED BREAD/BUNS/ROLLS,MAINSTREAM WHITE BREAD,910,National,GROCERY,518,20,51,"1,138.0000",34,1.5504,0.2331,1.7683,1.6950,0.6448,0.0135,0.1503,0.5078,1.0000,3.0000,6.0000
32239,961554,CARROTS,CARROTS MINI PEELED,69,Private,PRODUCE,514,20,51,889.0000,19,1.5102,0.2766,1.5603,1.5292,0.3521,0.0000,0.1832,0.5039,1.0000,3.0000,7.0000
26318,904360,VEGETABLES SALAD,HEAD LETTUCE,673,National,PRODUCE,502,20,51,812.0000,5,0.9797,0.0489,1.5259,1.5100,0.0438,0.0000,0.0499,0.4922,1.0000,3.0000,19.0000
16430,6534178,COUPON/MISC ITEMS,GASOLINE-REG UNLEADED,69,Private,KIOSK-GAS,492,11,49,"39,023,136.0000",492,0.0022,0.0003,7.1382,6.2459,1.0000,0.0000,0.1161,0.4824,"10,568.0000","25,011.0000","89,638.0000"


## 14. Eligible screen

A 90% coverage cut is too strong for a household panel. Start with stores, weeks, price support and `price_cv`.

In [16]:
eligible = builder.screen_eligible(product_stats)

eligible SKUs: 338
      product_code                 category                   sub_commodity  coverage_rate  n_stores  n_weeks  unique_prices  price_cv  \
7314       1082185           TROPICAL FRUIT                         BANANAS         0.9049        20       51            664    0.2708   
2805       1029743      FLUID MILK PRODUCTS           FLUID MILK WHITE ONLY         0.7314        20       51             83    0.1154   
36058       995242      FLUID MILK PRODUCTS           FLUID MILK WHITE ONLY         0.6353        20       51             64    0.2904   
34402       981760                     EGGS                  EGGS - X-LARGE         0.6157        20       51             61    0.1491   
9139       1106523      FLUID MILK PRODUCTS           FLUID MILK WHITE ONLY         0.6059        20       51             45    0.1182   
36098       995785              PEPPERS-ALL              PEPPERS GREEN BELL         0.5206        20       51             39    0.2497   
30969       951

## 15. Choose a coherent group

Prefer `SUB_COMMODITY_DESC` with ≥10 eligible SKUs (closer substitutes). If none, climb to `COMMODITY_DESC`. Pre-specified ranking: median coverage, then volume.

In [17]:
level, category, sub = builder.choose_product_group(eligible)
level, category, sub

sub-commodities with enough SKUs:
              category                   sub_commodity  n_products  median_coverage  median_price_cv  total_units
125        SOFT DRINKS  SFT DRNK 2 LITER BTL CARB INCL          10           0.1255           0.1916   3,515.0000
126        SOFT DRINKS  SOFT DRINKS 12/18&15PK CAN CAR          15           0.1078           0.2308   4,448.0000
42              CHEESE                 SHREDDED CHEESE          12           0.1025           0.1658   2,361.0000
8           BAG SNACKS            TORTILLA/NACHO CHIPS          10           0.0902           0.1574   1,421.0000
31   CANDY - CHECKLANE  CANDY BARS (SINGLES)(INCLUDING          14           0.0819           0.2714   2,667.0000
159             YOGURT          YOGURT NOT MULTI-PACKS          15           0.0804           0.1429   2,513.0000
chosen sub_commodity: SOFT DRINKS / SFT DRNK 2 LITER BTL CARB INCL


('sub_commodity', 'SOFT DRINKS', 'SFT DRNK 2 LITER BTL CARB INCL')

## 16. Shortlist ~30 candidates

The final 20 wait for `causal_data.csv` (promo coverage).

In [18]:
candidates = builder.shortlist_candidates(eligible, level, category, sub)

builder.selection = DunnSampleSelection(
    cutoff_week_id=config.selection_cutoff,
    grouping_level=level,
    category=category,
    sub_commodity=sub,
    core_stores=list(builder.core_stores),
    candidate_product_codes=candidates["product_code"].astype(str).tolist(),
    product_codes=[],
    n_eligible_in_group=int(len(candidates)),
    criteria={
        "min_store_weeks": config.min_store_weeks,
        "n_core_stores": config.n_core_stores,
        "min_stores": config.min_stores,
        "min_weeks": config.min_weeks,
        "min_unique_prices": config.min_unique_prices,
        "min_price_cv": config.min_price_cv,
        "min_promo_coverage": config.min_promo_coverage,
        "n_skus": config.n_skus,
    },
)
builder.selection.to_dict()

candidate SKUs: 10


{'cutoff_week_id': 51,
 'grouping_level': 'sub_commodity',
 'category': 'SOFT DRINKS',
 'sub_commodity': 'SFT DRNK 2 LITER BTL CARB INCL',
 'core_stores': ['367',
  '406',
  '361',
  '356',
  '381',
  '32004',
  '292',
  '427',
  '31782',
  '375',
  '372',
  '319',
  '321',
  '401',
  '333',
  '369',
  '422',
  '388',
  '439',
  '327'],
 'candidate_product_codes': ['1053690',
  '844165',
  '1092026',
  '916381',
  '893501',
  '1132770',
  '868764',
  '1076875',
  '839849',
  '1010190'],
 'product_codes': [],
 'n_eligible_in_group': 10,
 'joint_coverage_ge_8': None,
 'joint_coverage_all': None,
 'criteria': {'min_store_weeks': 45,
  'n_core_stores': 20,
  'min_stores': 5,
  'min_weeks': 35,
  'min_unique_prices': 8,
  'min_price_cv': 0.03,
  'min_promo_coverage': 0.2,
  'n_skus': 10}}

## 17. `causal_data.csv` in chunks

~36.8M rows. Do **not** `read_csv` the whole file. Filter to candidate products × core stores while streaming.

In [19]:
causal = builder.load_causal_for_candidates()
print(causal.shape)
print(causal["display"].value_counts(dropna=False).head(15))
print(causal["mailer"].value_counts(dropna=False).head(15))
causal.head()

causal subset: (4931, 5)
(4931, 5)
display
0    2511
7    1218
1     298
2     279
6     181
3     160
A      92
9      90
4      74
5      28
Name: count, dtype: Int64
mailer
D    1910
A    1739
0     551
H     504
L      89
J      76
F      41
X      21
Name: count, dtype: Int64


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,839849,292,9,6,A
1,839849,292,15,0,H
2,839849,292,16,0,H
3,839849,292,17,0,A
4,839849,292,24,0,A


## 18. Observed merchandising → `on_promo`

`display == "0"` / `mailer == "0"` → no feature. Any other code → display or mailer.

This is **not** a markdown proxy. Codes are categorical, not ordinal.

In [20]:
promo_weekly = builder.construct_on_promo(causal)
builder.promo_weekly = promo_weekly
del causal

print("on_promo rate:", promo_weekly["on_promo"].mean())
print("on_display   :", promo_weekly["on_display"].mean())
print("in_mailer    :", promo_weekly["in_mailer"].mean())
promo_weekly.head()

on_promo rate: 1.0
on_display   : 0.49691991786447637
in_mailer    : 0.8868583162217659


,PRODUCT_ID,STORE_ID,WEEK_NO,on_promo,on_display,in_mailer,product_code,store_code,week_id
0,839849,292,9,1,1,1,839849,292,9
1,839849,292,15,1,0,1,839849,292,15
2,839849,292,16,1,0,1,839849,292,16
3,839849,292,17,1,0,1,839849,292,17
4,839849,292,24,1,0,1,839849,292,24


## 19. Candidate panel

Left join. No row in `causal_data` → no feature → `on_promo = 0`.

In [21]:
candidate_panel = builder.weekly[
    builder.weekly["product_code"].isin(candidates["product_code"])
    & builder.weekly["store_code"].isin(builder.core_stores)
].copy()

candidate_panel = candidate_panel.merge(
    promo_weekly[
        ["product_code", "store_code", "week_id", "on_promo", "on_display", "in_mailer"]
    ],
    on=["product_code", "store_code", "week_id"],
    how="left",
    validate="one_to_one",
)
candidate_panel["promo_observed"] = candidate_panel["on_promo"].notna()
candidate_panel["on_display"] = candidate_panel["on_display"].fillna(0).astype("int8")
candidate_panel["in_mailer"] = candidate_panel["in_mailer"].fillna(0).astype("int8")
candidate_panel["on_promo"] = candidate_panel["on_promo"].fillna(0).astype("int8")
builder.candidate_panel = candidate_panel

print("promo observed share:", candidate_panel["promo_observed"].mean())
print("on_promo rate       :", candidate_panel["on_promo"].mean())
print("on_promo NaN count  :", candidate_panel["on_promo"].isna().sum())
candidate_panel.head()

promo observed share: 0.4141791044776119
on_promo rate       : 0.4141791044776119
on_promo NaN count  : 0


,store_code,product_code,week_id,units,sales_value,customer_spend,regular_value,retail_discount,coupon_discount,coupon_match,n_baskets,n_households,retail_discount_events,coupon_events,price,price_customer,price_regular,retail_discount_share,coupon_share,brand,department,style,category,sub_commodity,size,on_promo,on_display,in_mailer,promo_observed
0,32004,1053690,1,1.0000,1.2500,1.2500,1.5900,0.3400,0.0000,0.0000,1,1,1,0,1.2500,1.2500,1.5900,0.2138,0.0000,1208,GROCERY,National,SOFT DRINKS,SFT DRNK 2 LITER BTL CARB INCL,2LTR,0,0,0,False
1,32004,868764,1,1.0000,1.3400,1.3400,1.5900,0.2500,0.0000,0.0000,1,1,1,0,1.3400,1.3400,1.5900,0.1572,0.0000,103,GROCERY,National,SOFT DRINKS,SFT DRNK 2 LITER BTL CARB INCL,2 LTR,0,0,0,False
2,367,1053690,1,3.0000,3.7500,3.7500,4.7700,1.0200,0.0000,0.0000,1,1,1,0,1.2500,1.2500,1.5900,0.2138,0.0000,1208,GROCERY,National,SOFT DRINKS,SFT DRNK 2 LITER BTL CARB INCL,2LTR,0,0,0,False
3,367,1076875,1,1.0000,1.3400,1.3400,1.5900,0.2500,0.0000,0.0000,1,1,1,0,1.3400,1.3400,1.5900,0.1572,0.0000,2224,GROCERY,National,SOFT DRINKS,SFT DRNK 2 LITER BTL CARB INCL,2 LTR,0,0,0,False
4,375,893501,1,1.0000,1.2500,1.2500,1.5900,0.3400,0.0000,0.0000,1,1,1,0,1.2500,1.2500,1.5900,0.2138,0.0000,1208,GROCERY,National,SOFT DRINKS,SFT DRNK 2 LITER BTL CARB INCL,2 LTR,0,0,0,False


## 20. Promo coverage by UPC

`promo_coverage` is the causal match rate, not “weeks on promotion”.
Absence of a causal row is coded as `on_promo = 0`.

In [22]:
promo_coverage = builder.measure_promo_coverage()
promo_coverage.sort_values("promo_coverage", ascending=False)

selection = builder.finalize_skus_with_promo_support()
selection.to_dict()

  product_code  n_rows  promo_rows  promo_coverage
7       868764     119          63          0.5294
3      1092026     215         109          0.5070
2      1076875     103          51          0.4951
8       893501     132          60          0.4545
6       844165     274         123          0.4489
9       916381     163          70          0.4294
1      1053690     288         107          0.3715
0      1010190      72          22          0.3056
5       839849      82          20          0.2439
4      1132770     124          29          0.2339
  product_code  n_rows  promo_rows  promo_coverage
7       868764     119          63          0.5294
3      1092026     215         109          0.5070
2      1076875     103          51          0.4951
8       893501     132          60          0.4545
6       844165     274         123          0.4489
9       916381     163          70          0.4294
1      1053690     288         107          0.3715
0      1010190      72         

{'cutoff_week_id': 51,
 'grouping_level': 'sub_commodity',
 'category': 'SOFT DRINKS',
 'sub_commodity': 'SFT DRNK 2 LITER BTL CARB INCL',
 'core_stores': ['367',
  '406',
  '361',
  '356',
  '381',
  '32004',
  '292',
  '427',
  '31782',
  '375',
  '372',
  '319',
  '321',
  '401',
  '333',
  '369',
  '422',
  '388',
  '439',
  '327'],
 'candidate_product_codes': ['1053690',
  '844165',
  '1092026',
  '916381',
  '893501',
  '1132770',
  '868764',
  '1076875',
  '839849',
  '1010190'],
 'product_codes': ['1053690',
  '1092026',
  '844165',
  '916381',
  '893501',
  '868764',
  '1076875',
  '1132770',
  '839849',
  '1010190'],
 'n_eligible_in_group': 10,
 'joint_coverage_ge_8': 0.0043859649122807015,
 'joint_coverage_all': 0.0,
 'criteria': {'min_store_weeks': 45,
  'n_core_stores': 20,
  'min_stores': 5,
  'min_weeks': 35,
  'min_unique_prices': 8,
  'min_price_cv': 0.03,
  'min_promo_coverage': 0.2,
  'n_skus': 10}}

## 22. ICDN panel

Full horizon, frozen SKUs, rows with `q > 0` and `p > 0`.

Missing causal is `on_promo = 0`. The panel must contain both promo and non-promo weeks.

In [23]:
icdn_panel = builder.build_icdn_panel()
print(icdn_panel.shape)
print(icdn_panel.dtypes)
icdn_panel.head(12)

rows dropped for non-positive price: 0
Wrote ICDN panel /home/thebigmonster/Github/nn-elasticity-additional-work/data/dunnhumby/panel/dunnhumby_icdn_panel.parquet
(3216, 9)
store_code      string[python]
product_code    string[python]
week_id                  int16
price                  float32
units                  float32
on_promo                  int8
category        string[python]
brand           string[python]
style           string[python]
dtype: object


,store_code,product_code,week_id,price,units,on_promo,category,brand,style
0,32004,1053690,1,1.2500,1.0000,0,SOFT DRINKS,1208,National
1,32004,868764,1,1.3400,1.0000,0,SOFT DRINKS,103,National
2,367,1053690,1,1.2500,3.0000,0,SOFT DRINKS,1208,National
3,367,1076875,1,1.3400,1.0000,0,SOFT DRINKS,2224,National
4,375,893501,1,1.2500,1.0000,0,SOFT DRINKS,1208,National
5,388,839849,1,0.5000,3.0000,0,SOFT DRINKS,69,Private
6,292,1076875,2,0.8800,2.0000,0,SOFT DRINKS,2224,National
7,292,844165,2,0.8800,1.0000,0,SOFT DRINKS,103,National
8,333,1076875,2,0.8800,1.0000,0,SOFT DRINKS,2224,National
9,427,868764,2,0.8800,1.0000,0,SOFT DRINKS,103,National


## 23. Final checks

In [24]:
print("price min / units min :", icdn_panel["price"].min(), icdn_panel["units"].min())
print("on_promo values       :", sorted(icdn_panel["on_promo"].unique().tolist()))
print("n products            :", icdn_panel["product_code"].nunique())
print("n stores              :", icdn_panel["store_code"].nunique())
print("week_id range         :", icdn_panel["week_id"].min(), "→", icdn_panel["week_id"].max())
print("outputs:", sorted(p.name for p in OUT_DIR.iterdir()))
icdn_panel.groupby("product_code", observed=True).agg(
    n_obs=("units", "size"),
    n_weeks=("week_id", "nunique"),
    n_stores=("store_code", "nunique"),
    mean_units=("units", "mean"),
    promo_rate=("on_promo", "mean"),
)

price min / units min : 0.47 1.0
on_promo values       : [0, 1]
n products            : 10
n stores              : 20
week_id range         : 1 → 102
outputs: ['dunnhumby_candidate_panel.parquet', 'dunnhumby_icdn_panel.parquet', 'dunnhumby_product_diagnostics.csv', 'dunnhumby_selected_product_diagnostics.csv', 'dunnhumby_selected_skus.json', 'dunnhumby_transaction_audit.json', 'dunnhumby_weekly_master.parquet', 'dunnhumby_weekly_transaction_master.parquet', 'icdn']


,n_obs,n_weeks,n_stores,mean_units,promo_rate
product_code,,,,,
1010190,135,74,16,2.1185,0.3185
1053690,609,101,20,2.5567,0.3547
1076875,237,91,20,1.6709,0.5148
1092026,447,96,20,2.2662,0.5257
1132770,236,90,19,1.4153,0.1483
839849,126,70,16,2.9524,0.2460
844165,534,101,20,2.8052,0.4813
868764,249,92,20,3.4016,0.5341
893501,306,95,20,2.0000,0.4052
